In [ ]:
using Pipe
# using MarketTechnicals
# using Plots
# using TimeSeries
using DataFrames
# using StatsPlots
using CSV
# using RollingFunctions
using Query
# using GLMakie

using JSON
using DataFrames
using Dates
using Colors
using ColorSchemes
using Indicators
using JSONTables
using JSON3

import Colors
using Plots
using Interact
using Evolutionary

In [ ]:
json_data_folder = "/media/mu6mula/Data/Crypto-Data-Feed/freq-user-data/data/binance"

# "$(ba)_$(qa)-$(tfr).json"
function LoadOHLC(ba,qa,tfr)
    fname = "$json_data_folder/$(ba)_$(qa)-$(tfr).json"
    json_string = read(fname, String)
    json_data = JSON3.read(json_string)
    
    odf = DataFrame(ts = Int64[], open=Float64[], high=Float64[], low=Float64[], close=Float64[], volume=Float64[])
    for r in json_data push!(odf, r) end
    return odf        
end
# dfa.unix_time .= dfa.unix_time ./1000
# dfa.time .= unix2datetime.(dfa.unix_time)
ba,qa,tfr = "BTC","USDT","1d"
odf = LoadOHLC(ba,qa,tfr)

In [ ]:
plot(odf.close)

In [ ]:
function donch(hl::AbstractMatrix{T}; n::Int64=10, inclusive::Bool=true)::Matrix{Float64} where {T<:Real}
    @assert size(hl,2) == 2 "Argument `hl` must have exactly 2 columns."
    local lower::Array{T} = runmin(hl[:,2], n=n, cumulative=false, inclusive=inclusive)
    local upper::Array{T} = runmax(hl[:,1], n=n, cumulative=false, inclusive=inclusive)
    local middle::Array{T} = (lower .+ upper) ./ 2.0
    return [lower middle upper]
end
function shft(x::AbstractVector{T} where T<:Number, n::Integer, fillVal=NaN)
    if n == 0
        return x
    elseif n < 0
        return vcat(x[1-n:end], fill(fillVal, -n))
    else
        return vcat(fill(fillVal, n), x[1:end-n])
    end
end
function ichimoku(hlc::AbstractMatrix{T}; params=(10, 26, 26, 52))::Matrix{Float64} where {T<:Real}
    # Source: https://www.investopedia.com/terms/i/ichimokuchart.asp
    # TODO: Implement option to get forward looking Senkous
    @assert size(hlc,2) == 3 "Argument `hlc` must have exactly 3 columns."
    tenkan = donch(hlc[:,1:2]; n=params[1], inclusive=true)[:,2]
    kijun = donch(hlc[:,1:2]; n=params[2], inclusive=true)[:,2]
    senkoua = shft((tenkan + kijun) /2, params[3])
    senkoub = shft(donch(hlc[:,1:2]; n=params[4], inclusive=true)[:,2], params[3])
    # chikou = shft(hlc[:,3], params[5])
    return [tenkan kijun senkoua senkoub]
end

In [ ]:
2  in [4,3]

In [ ]:
function xstrategy(odf::DataFrame, xparams)
    tenN,kijN,senaN,senbN = xparams
    n = nrow(odf)
    lclose = log.(odf.close)
    ichi = ichimoku(Matrix(odf[:,[:high,:low,:close]]); params=(tenN,kijN,senaN,senbN))
    tenkan,kijun,senkA,senkB = [view(ichi, :, i) for i in 1:size(ichi, 2)]
    ###
    senkU = max.(senkA,senkB)
    senkL = min.(senkA,senkB)
    tBk = tenkan .> kijun
    maskLong = tBk .&& (odf.close .> senkU) 
    sigLong = (.! shft(maskLong,1,true)) .&& maskLong
    maskLongEx = (odf.close .<= senkU) 
    maskLongEx[end] = true

    isigLong = (1:n)[sigLong]
    trades = zeros(n)
    inpos = false
    for i in 1:n 
        if (!inpos) && sigLong[i]
        trades[i] = 1 
        inpos = true
        elseif (inpos) && maskLongEx[i]
            trades[i] = -1
            inpos = false
        end
    end
    if inpos trades[n] = -1 end
    ilong = findall(x->x==1, trades)
    iexit = findall(x->x==-1, trades)
    pnl = lclose[iexit] .- lclose[ilong]
    cpnl = cumsum(pnl)
    return ilong, iexit, pnl, cpnl
end


# xstrategy(odf)

###!!! Vectorized
# xlong = vcat(isigLong,[n])
# longPer = [xlong[il]:xlong[il+1]-1 for il in 1:length(xlong)-1]
# # longPer = [xlong[il]:n for il in 1:length(xlong)-1]
# ilongEx = [
#     # b[1]+findfirst(maskLongEx[b]) 
#     # findfirst(maskLongEx[b])
#     sum(maskLongEx[b]) 
#     # b
#     for b in longPer
    
# ]

In [ ]:
# vectorized backtesting
# xparams = [20,60,30,120]
function xstrategy2(odf::DataFrame, xparams)
    tenN,kijN,senaN,senbN = xparams
    n = nrow(odf)
    lclose = log.(odf.close)
    ichi = ichimoku(Matrix(odf[:,[:high,:low,:close]]); params=(tenN,kijN,senaN,senbN))
    tenkan,kijun,senkA,senkB = [view(ichi, :, i) for i in 1:size(ichi, 2)]
    ###
    senkU = max.(senkA,senkB)
    senkL = min.(senkA,senkB)
    tBk = tenkan .> kijun
    maskEnter = tBk .&& (odf.close .> senkU) 
    sigEnter = (.! shft(maskEnter,1,false)) .&& maskEnter

    maskExit = (odf.close .<= senkU) 
    maskExit[end] = true
    sigExit = (.! shft(maskExit,1,false)) .&& maskExit

    # isigLong = (1:n)[sigLong]
    aipos = zeros(n)
    inpos = false
    for i in 1:n 
        if (!inpos) && maskEnter[i]
        aipos[i] = 1 
        inpos = true
        elseif (inpos) && maskExit[i]
            aipos[i] = -1
            inpos = false
        end
    end
    # [collect(1:Int(length(itrades) / 2)).*2]
    # ienter = 
    itrades = findall(x->x!=0, aipos) 
    itrEnter = collect(1:Int(length(itrades) / 2)).*2 .- 1
    itrExit = collect(1:Int(length(itrades) / 2)).*2

    itrEnter = collect(1:Int(length(itrades) / 2)).*2 .- 1
    itrExit = collect(1:Int(length(itrades) / 2)).*2
    iEnter = itrades[itrEnter]
    iExit = itrades[itrExit]
    lpEnter = lclose[iEnter]
    lpExit = lclose[iExit]
    pnl = lpExit.-lpEnter
    dfTrades = DataFrame(iEnter=iEnter,iExit=iExit,lpEnter=lpEnter, lpExit=lpExit, pnl=pnl, cpnl=cumsum(pnl))
    return ichi, dfTrades
end


In [ ]:

xparams = [20,60,30,120]
xstrategy2(odf, xparams)

In [ ]:
# x0 = [20,60,120,30]
# # tenN,kijN,senaN,senbN = x0
# trunc.(x0)

In [ ]:


# function obj(x) 
#     pars = trunc.(x)
#     println("----------------")
#     println(pars)
#     -xstrategy(odf, pars)[end][end] 
#     # -1
# end
# # obj(params)
# ga = GA(populationSize=20)
# lower = [3, 3,3,3]
# upper = [200, 200, 200, 200]
# Evolutionary.optimize(obj, BoxConstraints(lower, upper), x0, ga, Evolutionary.Options(iterations=20))

In [ ]:
ba,qa,tfr = "ALGO","USDT","8h"

@time ichi, dft = xstrategy2(odf, xparams)

In [ ]:
ba,qa,tfr = "ETH","USDT","8h"
odf = LoadOHLC(ba,qa,tfr)
tenN,kijN,senAn,senbN = 20,60,30,120
xparams = [tenN,kijN,senAn,senbN]
ichi, dft = xstrategy2(odf, xparams)
# xstrategy2(odf, xparams=[tenN,kijN,senAn,senbN])
# ws,ww = 1100,500; we=ws+ww
# ws,we = 1,nrow(odf)
ws,we = 1,nrow(odf)

wdft = dft[(dft.iEnter .>= ws) .&& (dft.iExit .<= we),:] 

# iwEnter = [x for x in dft.iEnter if x in ws:we]
# iwExit = [x for x in dft.iExit if x in ws:we]
plot(ws:we,odf.close[ws:we],size=(1500,500);label="close", linewidth=2),
plot!(ws:we,ichi[ws:we,1];label="tenkan", linecolor=:blue)
plot!(ws:we,ichi[ws:we,2];label="kijun", linecolor=:red)
plot!(ws:we,ichi[ws:we,3], fillrange = ichi[ws:we,4], fillalpha = 0.35, fillcolor=:gray)
# plot!(twinx(),vcat([0],iwExit), vcat([0],dft.cpnl), linewidth=3, linecolor=:orange)
plot!(twinx(),vcat(ws-1,wdft.iExit), vcat([0],cumsum(wdft.pnl)), linewidth=3, linecolor=:orange)
vline!(wdft.iEnter, alpha=0.3)
vline!(wdft.iExit, linecolor=:red, alpha=0.3)
# plot!(ichi[ws:we,3:4], fillcolor = plot_color(:yellow, 0.3))

In [ ]:
dft

### ZigZag

In [61]:
function pct_change(input::AbstractVector{<:Number}, fillvalue=missing)
    res = @view(input[2:end]) ./ @view(input[1:end-1]) .- 1
    [fillvalue; res]
end

pct_change (generic function with 2 methods)

In [63]:
eps = 0.1

s = odf.close

dir = 1
extr = s[1]
iextr = 1
extrems = []
push!(extrems, [iextr, dir])
# pctdiffs = pct_change(s)
for i in 2:length(s)
    delta = s[i]/extr - 1
    ndelta = dir * delta
    # println("------------------")
    # println(i, " ", delta, " ", ndelta)
    if sign(ndelta) == 1
        # println("cnt: ", iextr," ", extr, " ", ndelta," ",dir)
        iextr,extr = i,s[i]            
    elseif eps < - ndelta
        # println("trn: ", iextr," ", extr, " ", ndelta," ",dir)
        push!(extrems, [iextr, dir])
        dir = -dir
        iextr,extr = i,s[i]
    end
end
length(s), length(extrems)

(4811, 131)

In [64]:
extrems

131-element Vector{Any}:
 [1, 1]
 [135, 1]
 [142, -1]
 [147, 1]
 [180, -1]
 [200, 1]
 [216, -1]
 [219, 1]
 [226, -1]
 [239, 1]
 ⋮
 [4457, -1]
 [4596, 1]
 [4620, -1]
 [4656, 1]
 [4661, -1]
 [4680, 1]
 [4706, -1]
 [4739, 1]
 [4747, -1]

In [ ]:
Pkg.add("Quandl")